In [1]:
import os
import torch
import shutil
from pathlib import Path

from datasets import load_dataset
from transformers import AutoModelForCausalLM, AutoTokenizer

from llmcompressor import oneshot
from llmcompressor.modifiers.quantization import GPTQModifier, QuantizationModifier

D:\coding\LG_AImers\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
from compressed_tensors.quantization import QuantizationArgs, QuantizationType
from llmcompressor.modifiers.transform import SpinQuantModifier, QuIPModifier

In [3]:
MODEL_ID = "./base_model"
OUT_DIR  = "./model"

DATASET_ID = "LGAI-EXAONE/MANTA-1M"
DATASET_SPLIT = "train"

NUM_CALIBRATION_SAMPLES = 256
MAX_SEQUENCE_LENGTH = 512

# Quantization
SCHEME = "W8A8"
TARGETS = ["Linear"]
IGNORE  = ["embed_tokens", "lm_head", "model.layers.0", "model.layers.1", "model.layers.28", "model.layers.29"]
KV_CACHE_SCHEME = QuantizationArgs(type=QuantizationType("float")) # float8

In [4]:
print("[INFO] 모델 로드 중...")

tokenizer = AutoTokenizer.from_pretrained(
    MODEL_ID,
    trust_remote_code=True,
)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.bfloat16
)

print("[INFO] 모델/토크나이저 로드 완료")

[INFO] 모델/토크나이저 로드 완료


In [5]:
print("[INFO] 캘리브레이션 데이터 로드 중...")

ds = load_dataset(
    DATASET_ID,
    split=f"{DATASET_SPLIT}[:{NUM_CALIBRATION_SAMPLES}]",
)

def preprocess(example):
    return {
        "text": tokenizer.apply_chat_template(
            example["conversations"],
            add_generation_prompt=True,
            tokenize=False)
    }

ds = ds.map(preprocess)

print("[INFO] 데이터 전처리 완료")

[INFO] 데이터 전처리 완료


In [6]:
print(ds)
print(model)

Dataset({
    features: ['id', 'conversations', 'complexity_label', 'text'],
    num_rows: 256
})
Exaone4ForCausalLM(
  (model): Exaone4Model(
    (embed_tokens): Embedding(102400, 2048, padding_idx=0)
    (layers): ModuleList(
      (0-29): 30 x Exaone4DecoderLayer(
        (self_attn): Exaone4Attention(
          (q_proj): Linear(in_features=2048, out_features=2048, bias=False)
          (k_proj): Linear(in_features=2048, out_features=512, bias=False)
          (v_proj): Linear(in_features=2048, out_features=512, bias=False)
          (o_proj): Linear(in_features=2048, out_features=2048, bias=False)
          (q_norm): Exaone4RMSNorm((64,), eps=1e-05)
          (k_norm): Exaone4RMSNorm((64,), eps=1e-05)
        )
        (mlp): Exaone4MLP(
          (gate_proj): Linear(in_features=2048, out_features=4096, bias=False)
          (up_proj): Linear(in_features=2048, out_features=4096, bias=False)
          (down_proj): Linear(in_features=4096, out_features=2048, bias=False)
          (ac

In [7]:
# norm 관련 레이어 이름 확인
for name, module in model.named_modules():
    if "norm" in name.lower() and "layers.0" in name:
        print(name, type(module))

model.layers.0.self_attn.q_norm <class 'transformers.models.exaone4.modeling_exaone4.Exaone4RMSNorm'>
model.layers.0.self_attn.k_norm <class 'transformers.models.exaone4.modeling_exaone4.Exaone4RMSNorm'>
model.layers.0.post_attention_layernorm <class 'transformers.models.exaone4.modeling_exaone4.Exaone4RMSNorm'>
model.layers.0.post_feedforward_layernorm <class 'transformers.models.exaone4.modeling_exaone4.Exaone4RMSNorm'>


In [8]:
for name, module in model.named_modules():
    if "layers.0." in name: # 모델에 따라 'h.0.' 또는 'blocks.0.' 일 수 있습니다.
        print(name)

model.layers.0.self_attn
model.layers.0.self_attn.q_proj
model.layers.0.self_attn.k_proj
model.layers.0.self_attn.v_proj
model.layers.0.self_attn.o_proj
model.layers.0.self_attn.q_norm
model.layers.0.self_attn.k_norm
model.layers.0.mlp
model.layers.0.mlp.gate_proj
model.layers.0.mlp.up_proj
model.layers.0.mlp.down_proj
model.layers.0.mlp.act_fn
model.layers.0.post_attention_layernorm
model.layers.0.post_feedforward_layernorm


In [9]:
print(f"[INFO] Quantization 시작 (scheme={SCHEME}, samples={NUM_CALIBRATION_SAMPLES}, max_len={MAX_SEQUENCE_LENGTH})...")

recipe = [
    QuantizationModifier(
        scheme=SCHEME,
        targets=TARGETS,
        ignore=IGNORE,
        kv_cache_scheme=KV_CACHE_SCHEME,
    )
]

oneshot(
    model=model,
    dataset=ds,
    recipe=recipe,
    max_seq_length=MAX_SEQUENCE_LENGTH,
    num_calibration_samples=NUM_CALIBRATION_SAMPLES,
)

print("[INFO] Quantization 완료")

BackendCompilerFailed: backend='inductor' raised:
RuntimeError: Cannot find a working triton installation. Either the package is not installed or it is too old. More information on installing Triton can be found at https://github.com/openai/triton

Set TORCH_LOGS="+dynamo" and TORCHDYNAMO_VERBOSE=1 for more information


You can suppress this exception and fall back to eager by setting:
    import torch._dynamo
    torch._dynamo.config.suppress_errors = True


In [ ]:
os.makedirs(OUT_DIR, exist_ok=True)

model.save_pretrained(OUT_DIR, save_compressed=True)
tokenizer.save_pretrained(OUT_DIR)

print(f"[INFO] 모델 저장 완료: {OUT_DIR}")

In [ ]:
zip_name = "baseline_FP4_kv_ca"
print(f"[INFO] {zip_name}.zip 생성 중...")

shutil.make_archive(
    base_name=zip_name,
    format="zip",
    root_dir=".",
    base_dir=OUT_DIR,
)

print(f"[INFO] 생성 완료: {zip_name}.zip")